# Step 2 — Build road network with duckOSM

**duckOSM** (part of [h3-routing-platform](https://github.com/khoshkhah/h3-routing-platform)) converts
an OSM PBF file into a routable **DuckDB** database.

## What it produces

A single `.duckdb` file with the following tables per transportation mode:

| Table | Content |
|---|---|
| `driving.edges` | Road segments — geometry, highway type, speed, travel cost, H3 index |
| `driving.nodes` | Network nodes (intersections) |
| `driving.edge_graph` | Adjacency list for routing |
| `driving.turn_restrictions` | Turn constraints from OSM relations |

## What this notebook does

1. Runs duckOSM on the filtered PBF (from notebook 1)
2. Displays network statistics (edge count, total km, breakdown by road type)
3. Visualizes the road network on an interactive map
4. Exports a preview CSV of the edges

**Prerequisites:**
- duckOSM installed: `pip install -e ../h3-routing-platform/tools/duckOSM`
- Filtered PBF file from notebook 1 (or any `.osm.pbf`)

In [ ]:
%%time
import duckdb
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium
import yaml
from pathlib import Path
from duckosm import DuckOSM, Config

# ── Configuration — only change NAME ──────────────────────────────────────
NAME = 'sodermalm'
# ─────────────────────────────────────────────────────────────────────────

CONFIG_PATH = Path(f'../config/{NAME}.yaml')   # generated by notebook 1
OUTPUT_DIR  = Path('../output');  OUTPUT_DIR.mkdir(exist_ok=True)

# Read DB path from the config YAML so there is no path duplication
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        cfg = yaml.safe_load(f)
    DB_PATH       = Path(cfg['output_path']) / f"{cfg['name']}.duckdb"
    PBF_PATH      = Path(cfg['pbf_path'])
    BOUNDARY_PATH = Path(cfg['boundary_path'])
else:
    # Fallback if notebook 1 hasn't been run yet
    DB_PATH       = Path(f'../db/{NAME}.duckdb')
    PBF_PATH      = Path(f'../map/{NAME}.osm.pbf')
    BOUNDARY_PATH = Path(f'../boundaries/{NAME}.geojson')
    print('WARNING: config not found — run notebook 1 first to generate it')

print(f'Config  : {CONFIG_PATH}')
print(f'PBF     : {PBF_PATH}')
print(f'DuckDB  : {DB_PATH}')
print(f'Exists  : {DB_PATH.exists()}  ', end='')
if DB_PATH.exists():
    print(f'({DB_PATH.stat().st_size / 1_048_576:.1f} MB)')
else:
    print('← run duckOSM cell below')

---
## Run duckOSM

duckOSM reads the PBF, filters roads by mode, builds directed edges, adds speeds,
calculates travel costs, and creates an H3 spatial index — all in one call.

You can configure it via a YAML file (`config/{name}.yaml`) or directly with `Config.from_args()`.
Both produce the same result.

In [2]:
%%time
# Use YAML config if it exists, otherwise build config programmatically
if CONFIG_PATH.exists():
    config = Config.from_yaml(str(CONFIG_PATH))
    print(f'Using config: {CONFIG_PATH}')
else:
    config = Config.from_args(
        pbf_path      = str(PBF_PATH),
        output_path   = str(DB_DIR),
        name          = NAME,
        boundary_path = str(BOUNDARY_PATH),
        modes         = MODES,
        build_graph          = True,
        h3_indexing          = True,
        h3_resolution        = H3_RESOLUTION,
        simplify             = True,
        process_speeds       = True,
        extract_restrictions = True,
        calculate_costs      = True,
    )
    print('Using programmatic config')

db_path = DuckOSM(config).run()
print(f'\nDone! Database: {db_path}  ({db_path.stat().st_size / 1_048_576:.1f} MB)')

⠹  Processing speeds ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━  79% 0:00:01

✓ Import completed in 1.13s

Import Summary

┏━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Mode    ┃ Nodes ┃  Edges ┃ Edge Pairs ┃  Time ┃
┡━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ driving │ 2,652 │  4,554 │      9,293 │ 0.24s │
│ walking │ 6,561 │ 15,414 │     41,323 │ 0.44s │
│ cycling │ 3,195 │  5,916 │     13,062 │ 0.27s │
└─────────┴───────┴────────┴────────────┴───────┘

Output size: 22.01 MB

Database Path: /home/kaveh/projects/osm-traffic-enrichment/map/sodermalm.duckdb

--------------------------------------------------


Done! Database: /home/kaveh/projects/osm-traffic-enrichment/map/sodermalm.duckdb  (22.0 MB)
CPU times: user 1.58 s, sys: 419 ms, total: 2 s
Wall time: 1.17 s


---
## Network statistics

Query the `driving.edges` table to understand what was built:
- Edge count and total network length by highway type
- Distribution of speed limits
- Oneway vs. bidirectional split

In [3]:
%%time
con = duckdb.connect(str(DB_PATH))
con.execute('LOAD spatial')

# ── Road type breakdown ───────────────────────────────────────────────────
hw_stats = con.execute("""
    SELECT highway,
           count(*)               AS edges,
           round(sum(length_m)/1000, 1) AS km,
           round(avg(maxspeed_kmh), 0)  AS avg_speed_kmh
    FROM driving.edges
    GROUP BY highway
    ORDER BY edges DESC
""").df()

total = con.execute("SELECT count(*) n, round(sum(length_m)/1000,1) km FROM driving.edges").df()

print('═' * 55)
print(f'  NETWORK SUMMARY  ({NAME})')
print('═' * 55)
print(f'  Total edges   : {total["n"].iloc[0]:>7,}')
print(f'  Total length  : {total["km"].iloc[0]:>7.1f} km')
print('═' * 55)
print('\nBreakdown by highway type:')
display(hw_stats)

CatalogException: Catalog Error: Table with name edges does not exist!
Did you mean "pg_catalog.pg_views"?

LINE 6:     FROM driving.edges
                 ^

In [ ]:
%%time
# ── Load edges as GeoDataFrame ────────────────────────────────────────────
df = con.execute("""
    SELECT edge_id, highway, name, oneway, length_m,
           maxspeed_kmh, cost_s,
           ST_AsText(geometry) AS wkt_geom
    FROM driving.edges
""").df()
con.close()

edges = gpd.GeoDataFrame(
    df,
    geometry=gpd.GeoSeries.from_wkt(df['wkt_geom']),
    crs='EPSG:4326',
)
lines = edges[edges.geometry.geom_type.isin(['LineString', 'MultiLineString'])]

# ── Color map by road category ────────────────────────────────────────────
ROAD_COLORS = {
    'motorway': '#e63946',      'motorway_link': '#e63946',
    'trunk':    '#f4a261',      'trunk_link':    '#f4a261',
    'primary':  '#2a9d8f',      'primary_link':  '#2a9d8f',
    'secondary':'#457b9d',      'secondary_link':'#457b9d',
    'tertiary': '#a8dadc',      'tertiary_link': '#a8dadc',
    'residential': '#6c757d',
    'service':  '#adb5bd',
}

bds = lines.total_bounds
center = [(bds[1] + bds[3]) / 2, (bds[0] + bds[2]) / 2]
m = folium.Map(location=center, zoom_start=13, tiles='OpenStreetMap')

folium.GeoJson(
    lines.__geo_interface__,
    style_function=lambda feat: {
        'color':  ROAD_COLORS.get(feat['properties'].get('highway', ''), '#cccccc'),
        'weight': 2,
    },
    tooltip=None, popup=None,
).add_to(m)
m

In [ ]:
%%time
# Export edges CSV for reference / downstream tools
csv_path = OUTPUT_DIR / f'{NAME}_edges.csv'
edges.drop(columns=['geometry', 'wkt_geom']).to_csv(csv_path, index=False)
print(f'Exported edges CSV → {csv_path}  ({len(edges):,} rows)')
print('Proceed to notebook 3 to fetch Mapbox traffic data.')